In [ ]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit, voer_pid_stap5_uit
import pandas as pd
import matplotlib.pyplot as plt

instellingen = dict(
    com_poort="COM4",
    beginhoek=115.4,  # Vul je gemeten ruwe sensorhoek in rust in.
    doelhoek=40.0,
    kp=2.0, ki=0.0, kd=0.0,  # Neem je eigen stap-4-instelling over.
    max_pwm=120,  # Maximale uitgang; neem je gekozen grens uit stap 4 over.
    meetduur=10.0,
    delta_t=0.02,
    mode="fixed",
    toon_metingen=True,
)


## 1. Referentiemeting zonder limiter

`max_pwm` is de maximale motoruitgang (voorbeeld: 120). Met `max_pwm=0` blijft de motor altijd uit. `max_pwm_per_seconde` bepaalt uitsluitend hoe snel de uitgang verandert. Gebruik dezelfde beginstand, doelhoek, voeding en PID-instellingen voor alle proeven. Iedere aanroep start een nieuwe proef en bewaart een eigen CSV.


In [ ]:
metingen_zonder, csv_zonder = voer_pid_stap5_uit(
    **instellingen,
)


## 2. Meting met begrensde PWM-verandering

Voer dezelfde proef uit met bijvoorbeeld 100 telwaarden per seconde. De limiter begrenst zowel stijgen als dalen. Bewaar de proef en herhaal daarna met bijvoorbeeld 50 en 200 telwaarden per seconde. Verander tussen deze proeven alleen de limiterinstelling.


In [ ]:
max_pwm_per_seconde = 100.0
metingen_met, csv_met = voer_pid_stap5_uit(
    **instellingen, max_pwm_per_seconde=max_pwm_per_seconde,
)


## 3. Vergelijken

Vergelijk de hoekrespons en de aangeboden PWM. Noteer insteltijd, overshoot, stationaire fout en hoe snel de PWM verandert. Herhaal elke instelling minstens drie keer. Bespreek welke waarde een bruikbare afweging geeft tussen vloeiende aansturing en snelheid.

De uitgang wordt afgerond op gehele PWM-telwaarden. Kleine afwijkingen van de berekende snelheid kunnen daarom door een verschil van een telwaarde ontstaan. De CSV bevat de werkelijk aangeboden PWM. De maximale hoek, heartbeat, meetduur en STOP blijven actief; die stops doorlopen de limiter niet.


In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
for pad, label in [(csv_zonder, "Zonder limiter"), (csv_met, "Met limiter")]:
    if pad is None:
        raise ValueError("Voer eerst beide metingen uit.")
    df = pd.read_csv(pad, comment="#")
    ax[0].plot(df["tijd_s"], df["hoek_deg"], label=label)
    ax[1].plot(df["tijd_s"], df["pwm"], label=label)
ax[0].axhline(instellingen["doelhoek"], color="black", linestyle="--", label="Doelhoek")
ax[0].set_ylabel("Werkhoek (graden)")
ax[1].set_ylabel("Aangeboden PWM")
ax[1].set_xlabel("Tijd sinds START (s)")
for a in ax:
    a.grid()
    a.legend()
fig.tight_layout()
plt.show()
